# Projet Python pour la Data Science : Influence des médailles remportées par la France aux Jeux Olympiques sur le nombre le licenciés sportifs en France.

_Autrices : Melissa MIGAN, Camille PEYTHIEUX-TALDIR, Romane PLUQUET_.

## Introduction

# to do

### Sommaire

#TODO

- [Introduction](#introduction)

- [I. Création et exploration de la base de données principale](#i-création-et-exploration-de-la-base-de-données-principale)

- [II. Visualisations et statistiques descriptives](#ii-visualisations-et-statistiques-descriptives)

- [Conclusion](#conclusion)

## I. Création et exploration de la base de données principale

Dans cette partie, l'objectif est d'importer et de travailler les différentes bases de données et de les joindre en une base de données exploitable. Après travail et nettoyage des données brutes (en accès public), nous utilisons trois bases de données :

| Nom de la base   | Description                                                                 | Source     | Mode d'extraction |
|-----------------|------------------------------------------------------------------------------|------------|-------------------|
| `data_medailles`  | Nombre de médailles reportées par la France aux Jeux Olympiques par sport et année (2016-2024). | Wikipedia  | Web scraping      |
| `data_licences`   | Nombre de licenciés sportifs en France par fédération, sexe, âge et département (2016-2024).       | Injep (Institut national de la jeunesse et de l'éducation populaire)          | CSV, Parquet      |
| `data_pop`        | Population départementale en France (recensements de 2016 et 2022).          | Insee      | API               |

Ces tables regroupent les données suivantes :
- La table `data_medailles` rassemble les médailles obtenues par la France aux Jeux Olympiques entre 2016 et 2024, c'est-à-dire aux JO de 2016, 2020 (qui ont eu lieu en 2021 à cause du Covid) et de 2024. 
- La table `data_licences` recense les effectif de licenciés par an entre 2016 et 2024. Elle présente également des effectifs par tranches d'âge et par genre.
- La table `data_pop` contient les populations départementales recensées en 2016 et en 2022. Elle nous permet de mener une étude des effectifs de licenciés par département, relativement à la population de ces derniers. Cette table n'étant utilisée que dans un unique graphique (dans un but de pondération), elle ne sera pas incluse dans notre base de données principale.

On importe les modules pour traiter les données, et les fonctions utilisées.

In [34]:
#%pip install -r requirements.txt

# Modules
#import os
import pandas as pd
import numpy as np
import pyarrow as pa

# Fonctions
from data import (
    gel_tableau_medailles,
    nettoyer_base,
    fusionner_bases,
    reorganiser_colonnes,
    normalisation_unicode,
    code_sport,
    code_dep,
    renommer_colonnes, 
    gel_licences,
    tableau_ratios_nr,
    melodi_extraction,
    code_dep_pop,
    clean_population,
    gel_population
)

### A. Récupération des données

#### 1. Médailles françaises aux Jeux Olympiques

Nous avons scrappé la page Wikipedia ["_France aux Jeux Olympiques_"](https://fr.wikipedia.org/wiki/France_aux_Jeux_olympiques) afin d'obtenir les tableaux des médailles (or, argent bronze) obtenues par la France lors des Jeux Olympiques, à l'aide de la fonction `tableau_scraper`.

In [35]:
a_figer = [["or", "M.C3.A9dailles_d.27or_3"],
           ["argent", "M.C3.A9dailles_d.27argent"],
           ["bronze", "M.C3.A9dailles_de_bronze"]]

#for duo in a_figer:
    #gel_tableau_medailles(duo[0], duo[1])

 Nous avons ensuite gelé les bases scrappées dans un souci de reproductibilité, au cas où la page Wikipédia soit modifiée. Nous avons ainsi obtenu trois tables :
- `data_or` : renseignant le nombre de médailles d'or obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_argent` : renseignant le nombre de médailles d'argent obtenues par la France aux Jeux Olympiques, par sport et par année,
- `data_bronze` : renseignant le nombre de médailles de bronze obtenues par la France aux Jeux Olympiques, par sport et par année.

In [36]:
data_or = pd.read_csv(f"data/data_medailles/data_or_jo.csv")
data_argent = pd.read_csv(f"data/data_medailles/data_argent_jo.csv")
data_bronze = pd.read_csv(f"data/data_medailles/data_bronze_jo.csv")

display(data_or.head())
display(data_argent.head())
display(data_bronze.head())

,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,45.0,1.0,2.0,1.0,0.0,2.0,3.0,...,0.0,2.0,2.0,3.0,1.0,0.0,2.0,0.0,5.0,1.0
1,2,NaN,Cyclisme,44.0,3.0,0.0,0.0,1.0,2.0,1.0,...,3.0,1.0,1.0,4.0,1.0,0.0,1.0,0.0,2.0,4.0
2,3,NaN,Judo,18.0,2.0,2.0,2.0,2.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,Équitation,14.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,2.0,0.0,0.0,0.0,1.0,NaN,NaN,1.0,NaN
4,4,NaN,Athlétisme,14.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,47.0,4.0,2.0,1.0,0.0,2.0,1.0,...,2.0,1.0,3.0,3.0,4.0,0.0,1.0,0.0,5.0,2.0
1,2,NaN,Cyclisme,30.0,3.0,0.0,0.0,3.0,3.0,1.0,...,2.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,1.0
2,2,NaN,Athlétisme,28.0,1.0,1.0,3.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,4.0,1.0
3,4,NaN,Natation,17.0,1.0,1.0,2.0,2.0,2.0,2.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
4,5,NaN,Aviron,15.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,NaN


,Place,Unnamed: 1,Sport,1896-2024,2024,2020,2016,2012,2008,2004,...,1936,1932,1928,1924,1920,1912,1908,1904,1900,1896
0,1,NaN,Escrime,38.0,2.0,1.0,1.0,0.0,0.0,2.0,...,1.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,5.0,0.0
1,2,NaN,Judo,34.0,6.0,3.0,1.0,5.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,NaN,Athlétisme,30.0,0.0,0.0,3.0,1.0,2.0,2.0,...,0.0,1.0,1.0,3.0,1.0,0.0,1.0,0.0,2.0,1.0
3,4,NaN,Cyclisme,28.0,3.0,2.0,1.0,0.0,1.0,1.0,...,2.0,1.0,0.0,2.0,1.0,0.0,2.0,0.0,1.0,1.0
4,5,NaN,Natation,22.0,2.0,0.0,1.0,1.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0


Nous avons ensuite nettoyé ces tables :
- nous avons supprimé les lignes et colonnes vides, ainsi que la colonne `Place`, qui ne servait que lors du scraping (pour être sûr de bien scraper toutes les lignes) ;
- nous avons retiré les années hors de notre période d'analyse (2016-2024) ;
- nous avons supprimé la colonne de total des médailles.

In [37]:
data_or_clean = nettoyer_base(data_or)
data_argent_clean = nettoyer_base(data_argent)
data_bronze_clean = nettoyer_base(data_bronze)

display(data_or_clean.head())
display(data_argent_clean.head())
display(data_bronze_clean.head())

,Sport,2024,2020,2016
0,Escrime,1.0,2.0,1.0
1,Cyclisme,3.0,0.0,0.0
2,Judo,2.0,2.0,2.0
3,Équitation,0.0,0.0,2.0
4,Athlétisme,0.0,0.0,0.0


,Sport,2024,2020,2016
0,Escrime,4.0,2.0,1.0
1,Cyclisme,3.0,0.0,0.0
2,Athlétisme,1.0,1.0,3.0
3,Natation,1.0,1.0,2.0
4,Aviron,0.0,1.0,0.0


,Sport,2024,2020,2016
0,Escrime,2.0,1.0,1.0
1,Judo,6.0,3.0,1.0
2,Athlétisme,0.0,0.0,3.0
3,Cyclisme,3.0,2.0,1.0
4,Natation,2.0,0.0,1.0


Suite à cela, nous avons joint ces trois tables afin d'obtenir la table `data_medailles_jo`, renseignant le nombre de médailles (toutes couleurs confondues) obtenues par la France aux Jeux Olympiques de 2016, 2020 et 2024.

Nous y avons ajouté trois variables :
- `code_sport`, un code permettant plus tard la jointure avec la table des licenciés sportifs en France,
- `total_medailles_2020`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2020,
- `total_medailles_2024`, une somme de toutes les médailles remportées lors des Jeux Olympiques de 2024.

In [38]:
data_medailles = fusionner_bases(data_or, data_argent, data_bronze)
display(data_medailles.sample(5))

,code_sport,sport,2024_or,2020_or,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
34,TIR,Tir,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0,1.0,1.0
18,JUD,Judo,2.0,2.0,2.0,2.0,3.0,2.0,6.0,3.0,1.0,5.0,8.0,10.0
10,ESD,Escalade,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0
11,ESC,Escrime,1.0,2.0,1.0,4.0,2.0,1.0,2.0,1.0,1.0,3.0,5.0,7.0
32,TEN,Tennis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### 2. Licenciés sportifs en France

Nous avons exploité les données de licenciés en France fournies par l'Injep, l'Institut national de la Jeunesse et de l'Education populaire. Les données se présentent sous forme brute comme un fichier CSV par an. Cependant, ces fichiers étant trop lourds pour être importés directement dans Git, nous optons pour une gestion des données par fichiers au format parquet. 


##### a. Construction de la base 

Notre but ici est de contruire une base de données au format long. Chaque base disposant déjà d'une colonne année, nous devons alors les concaténer pour obtenir la base au format désiré. Pour que l'opération se déroule correctement, nous réorganisons dans un premier temps les colonnes de chacune des bases, de sorte que chacune ait les mêmes colonnes dans le même ordre.

In [39]:
liste_fichiers = ["data/data_licences/Lics_2016_semidef.parquet",
                  "data/data_licences/Lics_2017_semidef.parquet", 
                  "data/data_licences/Lics_2018_semidef.parquet",
                  "data/data_licences/Lics_2019_def.parquet",
                  "data/data_licences/Lics_2020_def.parquet",
                  "data/data_licences/Lics_2021_def.parquet",
                  "data/data_licences/Lics_2022_def.parquet",
                  "data/data_licences/Lics_2023_semidef.parquet",
                  "data/data_licences/Lics_2024_semidef.parquet"]

data_licences = reorganiser_colonnes(liste_fichiers)
data_licences = pa.concat_tables(data_licences)

Afin d'éviter les problèmes de sélection de données, car nous disposons de variables dont les modalités sont textuelles, nous normalisons tous les caractères avec la norme unicode. Nous ajoutons ensuite plusieurs variables permettant un niveau d'analyse plus général que celui très fin proposé par la base de données ainsi qu'une harmonisation entre les différentes bases utilisées : 
- `code_sport` : catégorise les fédérations selon le sport pratiqué. Nous choisissons de catégoriser en "divers" (`code_sport` : DIV) les fédérations qui pratiquent un sport non-olympique. Ce code est le même que celui ajouté à la table des médailles. 
- `code_dep` : indique le département par son numéro seulement. 

In [40]:
data_licences = normalisation_unicode(data_licences)
data_licences = code_sport(data_licences)
data_licences = code_dep(data_licences, "Département")

Nous renommons ensuite les colonnes pour avoir des noms de variable sans majusucles ni accents. Nous ne gardons que les colonnes qui nous seront utiles pour la suite, à savoir celles concernant le nom de la fédération, l'année de recensement des licences, le sexe des licencié.e.s, les tranches d'âge (age, tranches fines et grandes fines), le nombre de licences annuelles, le code sport et le code département. Finalement, nous gelons la table dans un fichier parquet pour la réutiliser par la suite telle que construite ici. 

In [41]:
data_licences = renommer_colonnes(data_licences)
data_licences = data_licences[["federation","annee", "sexe", "age", "tranche_age","grande_tranche_age","licences_annuelles","code_sport","code_dep"]]
#gel_licences(data_licences)
display(data_licences.sample(5))
display(data_licences["federation"].describe())

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep
3312247,Fédération Française de Boxe,2020,F,13,c - de 10 à 14 ans,1 - Enfants (1-13),43,BOX,62
5789301,Fédération Française de Gymnastique,2023,H,4,a - de 1 à 4 ans,1 - Enfants (1-13),30,GYM,03
4686614,Fédération Française d'Éducation Physique et d...,2021,H,67,n - de 65 à 69 ans,4 - Seniors (56-99),3,DIV,47
7130816,Fédération Française de Vol Libre,2024,H,62,m - de 60 à 64 ans,4 - Seniors (56-99),6,DIV,26
4747829,Fédération Sportive et Gymnique du Travail,2021,H,36,h - de 35 à 39 ans,3 - Adultes (21-55),4,DIV,62


count                            7354671
unique                               118
top       Fédération Française de Tennis
freq                              156530
Name: federation, dtype: object

Nous obtenons une base de données comprenant plus de 7,35 millions de lignes. 118 fédérations sportives y sont recensées, et ce sur neuf années, de 2016 à 2024. La variable `code_sport` nous permet de catégoriser facilement les fédérations selon le sport pratiqué. Nous choisissons la modalité "DIV" pour les sport non-olympiques. Ainsi, 33 sports olympiques sont présents dans notre base de données. 

Nous disposons de plusieurs variables permettant de distinguer les licenciés selon des critères socio-démographiques. 
- trois variables d'âge :  
    - la variable `age` qui donne l'âge précis des licenciés, 
    - la variable `tranche_age` qui répertorie les licenciés selon 18 tranches d'âges "fines", par exemple ceux ayant de 10 à 14 ans,
    - la variable `grande_tranche_age` qui classe les licenciés selon cinq tranches d'âge plus larges, comme par exemple la catégorie Adultes (21-55 ans),
- la variable `sexe`, qui présente deux modalités, H ou F, et permet de classer les licenciés selon leur genre,
- la variable `code_dep`, qui renseigne le département dans lequel sont enregistrés les licenciés.

Finalement, notre variable d'intérêt est la variable `licences_annuelles`, que nous pouvons agréger selon toutes les variables présentes dans la base, en prenant les précautions nécessaires, détaillées dans la partie suivante.

##### b. Précautions : comparabilité dans le temps

Nos données de licences proviennent de fichiers distincts pour chaque année recensée. Ces fichiers sont de deux types :
- `semidef` pour les années 2016 à 2018 et 2023 à 2024, qui n'ont pas encore été "géocodées",
- `def` pour les années 2019 à 2022, qui sont "géocodées".

La documentation de ces données affirme que les données sont comparables dans le temps à condition d'être agrégées par fédération. Ainsi, comme notre code sport est directement dérivé des fédérations, l'agrégation selon la variable `code_sport` ne posera pas de problème de compabilité. Cependant, il est précisé que les "données par sexe, et/ou par âge, et/ou par département/région ne doivent pas être comparées dans le temps directement". Il faut dans ce cas là bien prendre en compte les effectifs non répartis (NR), c'est-à-dire qui n'ont pas pu être classés selon un département, une catégorie d'âge ou de genre, afin d'obtenir des résultats comparables dans le temps. 

Pour d'avoir une idée de l'ampleur de la non répartition géographique des effectifs de licences dans les données, nous calculons les ratios d'effectifs de licences géographiquement non répartis sur la totalité des effectifs de licences par an, ainsi que dans la base regroupant toutes les années. 

In [42]:
display(tableau_ratios_nr(data_licences, "code_dep"))

,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis code_dep,4.78 %,5.19 %,4.45 %,0.74 %,0.76 %,0.91 %,0.50 %,0.47 %,0.40 %,2.05 %


On constate alors que la proportion de licences non géographiquement réparties est bien plus importante pour les premiers fichiers `semidef`, c'est-à-dire de 2016 à 2018, alors que par la suite, cette proportion n'excède pas les 1%. Ainsi, dès que nous mènerons une analyse géographique, nous prendrons en compte dans l'élaboration et l'analyse des résultats le fait que, de 2016 à 2018, autour de 5% des effectifs de licences ne sont géographiquement pas attribués, dans la mesure où cette proportion est significativement importante. Nous montrons ensuite que, comme le suggère la précision dans la documentation sur le fait que certains fichiers soient "géocodés" ou non, la non répartition géographique est la plus importante dans notre jeu de données, grâce à un calcul de ratio similaire. 

In [43]:
display(tableau_ratios_nr(data_licences, "sexe"))
display(tableau_ratios_nr(data_licences, "age"))
display(tableau_ratios_nr(data_licences, "tranche_age"))
display(tableau_ratios_nr(data_licences, "grande_tranche_age"))

,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis sexe,2.43 %,2.39 %,0.90 %,2.79 %,0.00 %,0.00 %,0.00 %,0.00 %,0.00 %,0.97 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis tranche_age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


,2016,2017,2018,2019,2020,2021,2022,2023,2024,Global
Ratio de non répartis grande_tranche_age,1.16 %,0.69 %,2.96 %,0.34 %,0.25 %,0.31 %,0.03 %,0.01 %,0.01 %,0.65 %


En ce qui concerne l'âge et le sexe, on constate que la non répartition est globalement moins importante, mais toujours assez marquée de 2016 à 2018. La non répartition pour les variables d'âge en tranches est strictement égale à celle de la variable d'âge, puisque ces dernières découlent directement de la première. La non répartition en terme d'âge n'excède pas les 2% par an, et tend vers de très faibles valeurs à partir de 2019 (< 0,31%). La non répartition par sexe, elle, n'excède pas les 2,43% et est nulle de 2020 à 2024. Ainsi, bien que ce phénomène soit moins important pour l'âge et le sexe, nous le prendrons en compte dans nos analyses par âge et par sexe. 

#### 3. Population départementale en France

Dans l'optique de mener une analyse des effectifs de licenciés par départements, proportionnellement à leur population, nous récupérons les données de population départementale en France grâce à l'API Melodi de l'INSEE. Nous choisissons d'utiliser le jeu de données de la population de municipale et non de la population de référence, car la population municipale est comptabilisée tous les ans, tout comme nos données de licences. Ce choix permet une cohérence temporelle entre nos données. Par ailleurs, d'après l'INSEE, la population municipale est "désormais [...] la notion de population la plus utilisée en statistiques".

La documentation de l'API fournit directement le code permettant d'extraire les données voulues, que nous reprenons dans la fonction permettant l'extraction des données. L'URL d'extraction a été obtenu par visualisation de la base de données au niveau de précision désiré, c'est-à-dire au niveau départemental ; et permet de sélectionner directement les années d'intérêt (2016 à 2023). 

La base de données extraite comporte 800 lignes : la population pour les 96 départements de France métropolitaines et quatre départements d'Outre-mer (Guadeloupe, Martinique, Guyane, La Réunion), sur huit années. Les données n'étant pas encore disponibles pour l'année 2024, nous utiliserons celles de 2023 pour des analyses relatives à la population en 2024.

In [44]:
url_api = "https://api.insee.fr/melodi/data/DS_POPULATIONS_HISTORIQUES?TIME_PERIOD=2016&TIME_PERIOD=2017&TIME_PERIOD=2018&TIME_PERIOD=2019&TIME_PERIOD=2020&TIME_PERIOD=2021&TIME_PERIOD=2022&TIME_PERIOD=2023&GEO=DEP"
data_pop = melodi_extraction(url_api)
display(data_pop.sample(5))

Jeu de données : DS_POPULATIONS_HISTORIQUES 
Titre : Populations municipales de 2016 à 2023 


,GEO,FREQ,TIME_PERIOD,POPREF_MEASURE,OBS_VALUE_NIVEAU
162,2025-DEP-90,A,2016,PMUN,144089.0
649,2025-DEP-05,A,2019,PMUN,141220.0
499,2025-DEP-21,A,2016,PMUN,533213.0
776,2025-DEP-77,A,2022,PMUN,1452399.0
122,2025-DEP-71,A,2019,PMUN,551493.0


Nous ne sélectionnons que les variables utiles pour notre analyse, c'est-à-dire celles correspondant au département, à l'année et à la population comptablisée dans le département, puisque les variables de fréquence de mesure et de type de mesure sont unimodales. Nous renommons également les colonnes avec le même format que pour les bases présentées précédemment. Finalement, nous ajoutons le même code département que dans les autres bases, à partir de la variable de département. 

In [45]:
display(data_pop["FREQ"].unique())
display(data_pop["POPREF_MEASURE"].unique())

array(['A'], dtype=object)

array(['PMUN'], dtype=object)

In [46]:
data_pop = data_pop[["GEO", "TIME_PERIOD", "OBS_VALUE_NIVEAU"]]
data_pop.columns = ["departement", "annee", "population"]
data_pop = code_dep_pop(data_pop, "departement")

Nous nettoyons ensuite cette base, en forçant les types des variables (année en entiers, département en variables textuelles), en supprimant les départements non cartographiables et en les triant par ordre croissant. Nous gelons ensuite les données dans un fichier CSV. 

In [47]:
data_pop_clean = clean_population(data_pop)
#gel_population(data_pop_clean)
data_pop_clean.sample(5)

,departement,annee,population,code_dep
189,2025-DEP-25,2016,538549.0,25
591,2025-DEP-72,2020,566993.0,72
194,2025-DEP-26,2017,511553.0,26
395,2025-DEP-49,2016,810934.0,49
113,2025-DEP-15,2023,144196.0,15


### B. Jointure des tables

Nous avons joint les tables médailles et licences pour pouvoir étudier en détail l'effet de remporter des médailles aux Jeux Olympiques sur l'évolution du nombre de licenciés sportifs. Nous avons joint par la gauche en utilisant la clé `code_sport` pour ne pas démultiplier le nombre de lignes dans notre DataFrame : nous avons un unique code sport par ligne dans notre table médailles.

In [48]:
data_complet = pd.merge(data_licences, data_medailles, how='left', on="code_sport")
data_complet.sample(5)
#data_complet["code_sport"].unique()

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep,sport,...,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
1527758,Union Française des Œuvres Laïques d'Éducation...,2017,F,88,q - de 80 à 99 ans,4 - Seniors (56-99),1,DIV,55,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4429944,Fédération Française de Danse,2021,F,34,g - de 30 à 34 ans,3 - Adultes (21-55),7,DIV,75,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214314,Fédération Française de Tennis,2016,F,16,d - de 15 à 19 ans,2 - Jeunes (14-20),36,TEN,07,Tennis,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2133036,Fédération Française de Motocyclisme,2018,H,59,l - de 55 à 59 ans,4 - Seniors (56-99),1,DIV,11,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2661686,Fédération Française de Tennis,2019,F,36,h - de 35 à 39 ans,3 - Adultes (21-55),56,TEN,94,Tennis,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### C. Contrôle de la qualité des données

#### 1. Structure de la base

In [49]:
data_complet.shape

(7354671, 22)

In [50]:
data_complet.dtypes

federation               object
annee                     int64
sexe                     object
age                      object
tranche_age              object
grande_tranche_age       object
licences_annuelles        int64
code_sport               object
code_dep                 object
sport                    object
2024_or                 float64
2020_or                 float64
2016_or                 float64
2024_argent             float64
2020_argent             float64
2016_argent             float64
2024_bronze             float64
2020_bronze             float64
2016_bronze             float64
total_medailles_2016    float64
total_medailles_2020    float64
total_medailles_2024    float64
dtype: object

La base de données contient plus de sept millions d’observations et décrit les licences sportives selon l’année, le département du club sportif, le sport, le sexe, l’âge, et les médailles remportées aux Jeux Olympiques selon les sports. Les types des variables sont cohérents avec leur interprétation. Puisque pandas reconnait les _strings_ comme des _objects_, il est normal que certaines colonnes (comme `sexe` par exemple) soient indiquées être des objets alors qu'elles sont censées être des chaînes de caractères.

#### 2. Valeurs manquantes

In [51]:
na_table = (
    data_complet.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("taux_manquants")
)

na_table

,taux_manquants
2016_or,0.600609
2016_bronze,0.600609
2016_argent,0.600609
2024_bronze,0.571935
2024_argent,0.571935
2024_or,0.571935
2020_or,0.554199
total_medailles_2020,0.554199
total_medailles_2016,0.554199
2020_bronze,0.554199


Les valeurs manquantes concernent principalement les données issues de la tables `data_medailles`, et sont liées à la jointure par la gauche. Les valeurs manquantes concernant la variable géographique (`code_dep`, soit les départements) sont causées par les clubs sportifs de l'étranger et par les départements non répartis.

In [52]:
data_complet[data_complet["code_dep"].isna()]

,federation,annee,sexe,age,tranche_age,grande_tranche_age,licences_annuelles,code_sport,code_dep,sport,...,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
7237,Fédération Française d'Athlétisme,2016,F,7,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7238,Fédération Française d'Athlétisme,2016,F,8,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7239,Fédération Française d'Athlétisme,2016,F,9,b - de 5 à 9 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7240,Fédération Française d'Athlétisme,2016,F,10,c - de 10 à 14 ans,1 - Enfants (1-13),3,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
7241,Fédération Française d'Athlétisme,2016,F,11,c - de 10 à 14 ans,1 - Enfants (1-13),1,ATH,NaN,Athlétisme,...,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7344973,Union Nationale du Sport Scolaire (UNSS),2024,H,17,d - de 15 à 19 ans,2 - Jeunes (14-20),170,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344974,Union Nationale du Sport Scolaire (UNSS),2024,H,18,d - de 15 à 19 ans,2 - Jeunes (14-20),16,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344975,Union Nationale du Sport Scolaire (UNSS),2024,H,19,d - de 15 à 19 ans,2 - Jeunes (14-20),1,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7344976,Union Nationale du Sport Scolaire (UNSS),2024,H,33,g - de 30 à 34 ans,3 - Adultes (21-55),2,DIV,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 3. Cohérence des valeurs

##### a. Années observées

In [53]:
data_complet["annee"].min(), data_complet["annee"].max()

(np.int64(2016), np.int64(2024))

Les années couvertes par la base (2016-2024) sont cohérentes avec le périmètre temporel de l’étude.

##### b. Effectifs de licenciés

In [54]:
data_complet["licences_annuelles"].describe()

count    7.354671e+06
mean     1.939609e+01
std      1.488199e+02
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.100000e+01
max      1.518910e+05
Name: licences_annuelles, dtype: float64

In [55]:
(data_complet["licences_annuelles"] < 0).sum()

np.int64(0)

Aucun effectif négatif n’est observé. Les ordres de grandeur des effectifs sont cohérents avec des données de licences sportives.

#### 4. Unicité des observations

In [56]:
data_complet.duplicated().sum()

np.int64(2006)

Ces doublons stricts représentent environ 0,03% de notre base de données, soit une proportion négligeable. Nous les supprimons afin de garantir l'unicité des observations, sans impact significatif sur les analyses.

In [57]:
data_complet = data_complet.drop_duplicates()

### D. Export de la base finale

Nous avons ensuite exporté cette table dans le dossier `data`.

## TODO

In [58]:
#data_complet.to_parquet("data_complet.parquet")


#OUTPUT_DIR.mkdir(exist_ok=True)

#musees.to_csv(OUTPUT_DIR / "musees.csv", index=False)
#frequentation_annuelle.to_csv(OUTPUT_DIR / "frequentation_annuelle.csv", index=False)
#freq_excel_long.to_csv(OUTPUT_DIR / "frequentation_excel_long.csv", index=False)
#df_modele_clean.to_csv(OUTPUT_DIR / "df_modele_musees.csv", index=False)

#print("Fichiers exportés dans :", OUTPUT_DIR.resolve())


## II. Visualisations et statistiques descriptives

In [59]:
from fonctions_graphiques import (
    charger_donnees,
    widget_licences_par_sport,
    widget_graphique_licences_et_medailles,
    classement_sports_medailles,
    croissance_licencies_post_jo,
    widgets_evolution_licencies,
    widgets_evolution_licences_tranches_grande_age
)

In [60]:
(data_complet_2, gdf_dep, pop) = charger_donnees(
    data_complet_path="data/data_complet.parquet",
    geojson_path="departements.geojson",
    population_path="data/data_population/population_dept.csv",
)

### A(?). Exploration "naïve" de la base

In [61]:
widget_licences_par_sport(data_complet)

Dropdown(description='Sports :', options=('all', 'Athlétisme', 'Aviron', 'Badminton', 'Basket-ball', 'Boxe', '…

Output()

(Dropdown(description='Sports :', options=('all', 'Athlétisme', 'Aviron', 'Badminton', 'Basket-ball', 'Boxe', 'Canoë-kayak', 'Cyclisme', 'Escalade', 'Escrime', 'Football', 'Golf', 'Gymnastique', 'Haltérophilie', 'Handball', 'Hockey sur gazon', 'Judo', 'Karaté', 'Lutte', 'Natation', 'Pentathlon moderne', 'Rugby à 7', 'Skateboard', 'Surf', 'Taekwondo', 'Tennis', 'Tennis de table', 'Tir', "Tir à l'arc", 'Triathlon', 'Voile', 'Volley-ball', 'Équitation'), value='all'),
 Output())

Nous pouvons remarquer une baisse marquée du nombre de licenciés sportifs en 2021, tous sports confondus. Cette baisse est liée à la pandémie de Covid 19 et aux mesures sanitaires prises en France, qui ont fortement restreint la pratique sportive. Il nous faudra inclure cet évènement exceptionnel dans nos analyses et modélisations.

### A. Medailles et licenciés

Nous essayons ici d'observer les conséquences immédiates du nombre de médailles remportées aux Jeux Olympiques sur le nombre de licenciés sportifs.

Dans un premier temps, il s'agit d'observer quels sports ont remporté le plus de médailles olympiques grâce à la fonction `classement_sports_medailles`.

In [62]:
classement_sports_medailles(data_complet, 'all')

,sport,total_medailles,total_or,total_argent,total_bronze,score_pondere
0,Judo,23.0,6.0,7.0,10.0,42.0
1,Escrime,15.0,4.0,7.0,4.0,30.0
2,Natation,11.0,4.0,4.0,3.0,23.0
3,Cyclisme,12.0,3.0,3.0,6.0,21.0
4,Boxe,9.0,2.0,4.0,3.0,17.0
5,Voile,8.0,1.0,3.0,4.0,13.0
6,Athlétisme,8.0,0.0,5.0,3.0,13.0
7,Canoë-kayak,6.0,2.0,3.0,1.0,13.0
8,Handball,5.0,2.0,3.0,0.0,12.0
9,Équitation,6.0,2.0,2.0,2.0,12.0


commentaire : TODO

Les Jeux Olympiques de 2020 ayant eu lieu en 2021 en raison de la pandémie de Covid 19, nous avons décidé de définir l'année des médailles des JO 2020 en 2021 sur les graphiques.

In [63]:
widget_graphique_licences_et_medailles(data_complet)

Dropdown(description='Sport :', options=('all', 'Athlétisme', 'Aviron', 'Badminton', 'Basket-ball', 'Boxe', 'C…

Output()

Nous observons que l'obtention de médailles olympiques, notamment en or, affecte visiblement le nombre de licenciés sportifs dans certaines disciplines (volley ball, rugby, handball...). Cependant, cette observation n'est pas vraie pour tous les sports (judo, tir à l'arc...). Pour comprendre ces différences, nous nous intéresseront plus en détail aux caractéristiques des vainqueurs olympiques et des licenciés : sexe, âge.

Nous tentons maintenant de voir quels sports ont été le plus affectés par les Jeux Olympiques : lesquels ont-ils le plus vu leur nombre de licenciés augmenter dans les deux années suivant les Jeux Olympiques ? Nous choisissons de voir ce taux de croissance à t+2 pour éviter au maximum de percevoir l'effet du Covid 19.

In [64]:
display(croissance_licencies_post_jo(data_complet, 2016, 2).head())
display(croissance_licencies_post_jo(data_complet, 2020, 2).head())

,sport,annee_jo,annee_licences_t,licences_t,licences_t_plus_2,taux_croissance
0,Canoë-kayak,2016,2016,43369.0,57579.0,32.765339
1,Boxe,2016,2016,51623.0,59873.0,15.981249
2,Pentathlon moderne,2016,2016,1809.0,2020.0,11.663903
3,Tir,2016,2016,201574.0,224025.0,11.137845
4,Escrime,2016,2016,53627.0,56819.0,5.952226


,sport,annee_jo,annee_licences_t,licences_t,licences_t_plus_2,taux_croissance
0,Volley-ball,2020,2021,83245.0,147791.0,77.537390
1,Taekwondo,2020,2021,30184.0,50071.0,65.885900
2,Handball,2020,2021,340974.0,531864.0,55.983741
3,Karaté,2020,2021,166076.0,243970.0,46.902623
4,Judo,2020,2021,368661.0,529244.0,43.558445


### B. Evolution du nombre de licenciés

In [65]:
widgets_evolution_licencies(data_complet, gdf_dep)

Dropdown(description='Année 1 :', options=(np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.…

Dropdown(description='Année 2 :', index=1, options=(np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2…

Dropdown(description='Sport :', options=('all', 'Athlétisme', 'Aviron', 'Badminton', 'Basket-ball', 'Boxe', 'C…

Output()

In [66]:
widgets_evolution_licences_tranches_grande_age(data_complet)

Dropdown(description='Tranche :', options=('all', '1 - Enfants (1-13)', '2 - Jeunes (14-20)', '3 - Adultes (21…

Output()